## Reinforcement Learning

    -> MDPs (mathematical model)
        (S, A, R, y) or (S, A, P, R, γ)

    → Bellman equations
       → Dynamic Programming (DP)

    → Model-Based (Planning/DP -> when model is known)(solves Bellman exactly)
        → Value Iteration
        → Policy Iteration
            → Policy Evaluation
            → Policy Improvement

    → Model-Free (Learning -> when model is unknown)(approximates Bellman from samples)
        → Monte Carlo
        → Temporal Difference (TD)
            → SARSA (on-policy)
            → Q-learning (off-policy)

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

In [12]:
class gridworld:
    def __init__(self, size=5):
        self.size = size
        self.state = (0, 0) # Starting position
        self.walls = {(1, 1)}
        self.terminal_rewards = {(0, 4): -100, 
                                 (4, 0): -100, 
                                 (4, 4):  100} # Terminal states with rewards

    def is_terminal(self, state):
        return state in self.terminal_rewards

    def transition(self, state, action):
        if state in self.terminal_rewards:
            return state, 0, True
        
        x, y = state

        if action == 'up' and x > 0:
            x -= 1
        elif action == 'down' and x < self.size - 1:
            x += 1
        elif action == 'left' and y > 0:
            y -= 1
        elif action == 'right' and y < self.size - 1:
            y += 1
            
        candidate_state = (x, y)

        # Check for walls
        if candidate_state in self.walls:
            next_state = state
        else:
            next_state = candidate_state

        if next_state in self.terminal_rewards:
            reward = self.terminal_rewards[next_state]
            done = True
        else:
            reward = -10
            done = False

        return next_state, reward, done

### Model-Based

In [13]:
def value_iteration(env, states, actions, gamma, theta=0.001):
    V_table = {s: 0 for s in states}
    for s, r in env.terminal_rewards.items():
        V_table[s] = r
    history = [V_table.copy()]

    while True:
        delta = 0
        new_V = V_table.copy()
        for state in states:
            if env.is_terminal(state):
                continue

            action_values = []

            for action in actions:
                next_state, reward, _ = env.transition(state, action)
                v = reward + gamma * V_table[next_state]
                action_values.append(v)
    
            new_V[state] = max(action_values)
            delta = max(delta, abs(V_table[state] - new_V[state]))

        V_table = new_V
        history.append(V_table.copy()) 
        if delta < theta:
            break
    return history

# MDPs
env = gridworld()

# States and Actions space
states = [(i, j) for i in range(env.size) for j in range(env.size) if (i,j) not in env.walls]
actions = ['up', 'down', 'left', 'right']
gamma = 0.95

history = value_iteration(env, states, actions, gamma)

In [ ]:
def interactive_plot(iteration):
    V = history[iteration]
    size = env.size
    grid = np.zeros((size, size))

    for (i, j), v in V.items():
        if (i, j) in env.walls:
            grid[i, j] = np.nan 
        else:
            grid[i, j] = v

    plt.figure(figsize=(5, 5))

    # Use cmap to handle NaN as a white
    cmap = plt.cm.coolwarm.copy()
    cmap.set_bad(color='white')

    im = plt.imshow(grid, cmap=cmap, origin='upper')

    plt.colorbar(im, fraction=0.046, pad=0.04)

    for i in range(size):
        for j in range(size):
            if (i, j) in env.walls:
                continue  # Dont write on walls
            else:
                plt.text(j, i, 
                         f"{grid[i, j]:.2f}",
                         ha='center',
                         va='center',
                         fontsize=11,
                         color='black')

    plt.xticks(range(size))
    plt.yticks(range(size))
    plt.subplots_adjust(left=0, right=1, top=0.9, bottom=0)
    plt.title(f"Value Iteration — Iteração {iteration}")
    plt.tight_layout()
    plt.show()

print(interact(interactive_plot, iteration=IntSlider(min=0, max=len(history)-1, step=1, value=0)))

interactive(children=(IntSlider(value=0, description='iteration', max=9), Output()), _dom_classes=('widget-int…

<function interactive_plot at 0x000001BACC115120>
